# 부록 — 모델 행동 A/B 실험 3종 (`run_cc_harness.ipynb` §2 해설의 근거 실측)

메인 노트북 §2의 세 데모(2-2 병렬 emit · 2-3 게이트 충돌 · 2-5 위임)는 "장치는 §1에서 검증됐고,
모델이 그 장치를 실전에서 쓰는지는 실행마다 다르다"고 설명한다. 이 부록은 그 "실행마다 다르다"를
수치로 만든 실험이다 — 각 조건 3롤씩, 개입은 최소로.

| 실험 | 대상 데모 | A | B | 판정 지표 |
|---|---|---|---|---|
| E1 | 2-5 위임 | 시스템 프롬프트 당시 기본 (도구명 없는 우회 문구) | 위임 정책 한 문장에 **도구명(agent_search)만 명시** | agent_search 위임 발생 여부 |
| E2 | 2-2 병렬 emit | 현재 그대로 | 병렬 처방 뒤에 **"미루지 말라" 한 문장 추가** | `📦 파티션`(병렬 emit) 발생 여부 |
| E3 | 2-3 게이트 충돌 | 모델 = gpt-5.4-mini | 모델 = gpt-5-nano (프롬프트 불변) | read 없이 edit하다 게이트1에 걸리는 충돌 발생 여부 |

E1의 B는 OpenAI 공식 가이드의 권고 — *"Use the system prompt to describe when (and when not)
to use each function. Generally, tell the model exactly what to do."* (Function Calling 가이드) —
를 최소 단위로 적용해 본 것이다: 문장 구조는 그대로 두고 "맡길 수 있는 도구가 있으면" 부분만
"agent_search 도구에"로 바꿨다.

> **주의**: 아래 셀들의 출력은 2026-07-28 실측 1회분을 그대로 실은 것이다 (동일 코드의 프로브
> 스크립트 실행 기록). 다시 실행하면 API 비용이 들고, 수치는 롤에 따라 달라진다 — 이 실험의
> 결론 자체가 "모델 행동은 확률적"이므로 그게 정상이다. 메인 노트북 §2 해설이 인용하는 수치는
> 이 기록 기준이다.

In [1]:
import contextlib, io, re, sys
from collections import Counter

from cc_harness import Session
from cc_harness.prompts import DELEGATION_POLICY, PROMPT_PARALLEL

MINI = "gpt-5.4-mini"
N = 3

# B 변형 두 가지 — 둘 다 '한 문장' 수준의 최소 개입
DELEG_B = ("프로젝트 전반을 훑어야 하는 열린 조사는 검색 도구를 여러 번 반복하지 말고 "
           "agent_search 도구에 통째로 위임하세요.")
PAR_EXTRA = (" 이미 주어진 정보로 인자를 채울 수 있는 호출은 뒤 사이클로 미루지 말고 "
             "같은 응답에 포함하세요.")

Q_DELEG = ("이 프로젝트 전반의 예외 처리 방식(try/except, raise 등)이 어떤지 조사해서 요약해줘. "
           "나는 결과 요약만 보면 돼.")
Q_RENAME = "authenticate 라는걸 찾아서 이름을 verify_password로 바꿔줘."
Q_MENTION = "@/project/src/app/config.py 설정을 확인하고 배포 준비를 해줘. 필요한 수정이 있으면 직접 해줘."


def roll(q, model=MINI, patch=None, max_rounds=16):
    # 세션을 새로 만들고 (patch가 있으면 시스템 프롬프트만 최소 변형) 질문 1회 실행.
    # 출력은 버퍼로 삼켜서 판정 지표만 뽑는다.
    s = Session(model=model)
    if patch:
        s.system_prompt = patch(s.system_prompt)
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        s.ask(q, max_rounds=max_rounds)
    return s, buf.getvalue()


def summarize(out):
    return {
        "cycles": len(re.findall(r"═══ 사이클", out)),
        "delegated": "tool_invoke→agent_search" in out,
        "partition": "📦 파티션" in out,
        "gate1": out.count("아직 읽지 않았습니다"),
        "tools": dict(Counter(re.findall(r"🔧 ([\w→_]+)\(", out))),
    }


results = {}

## E1 — 위임 (2-5): 정책 문구에 도구명을 박으면 위임이 나오는가

실측 당시 시스템 프롬프트의 위임 정책은 도구 이름 없이 우회적이었다
("…조사를 통째로 맡길 수 있는 도구가 있으면 그쪽에 위임하세요").
B는 이 한 문장에서 그 부분만 "agent_search 도구에 통째로 위임하세요"로 바꾼다.

In [2]:
print("=== E1 · 2-5 위임 (시스템 프롬프트 정책 문구 A/B) ===")
for arm, patch in [("A_기본", None),
                   ("B_도구명_명시", lambda sp: sp.replace(DELEGATION_POLICY, DELEG_B))]:
    for i in range(N):
        s, out = roll(Q_DELEG, patch=patch)
        r = summarize(out)
        results.setdefault("E1_위임", {}).setdefault(arm, []).append(r)
        print(f"  {arm} #{i+1}: 위임={r['delegated']} 사이클={r['cycles']} {r['tools']}")

=== E1 · 2-5 위임 (시스템 프롬프트 정책 문구 A/B) ===
  A_기본 #1: 위임=False 사이클=4 {'glob_files': 1, 'grep_files': 1, 'read_file': 10}
  A_기본 #2: 위임=False 사이클=3 {'glob_files': 1, 'grep_files': 1, 'read_file': 9}
  A_기본 #3: 위임=False 사이클=5 {'glob_files': 1, 'grep_files': 4, 'read_file': 15}
  B_도구명_명시 #1: 위임=False 사이클=4 {'glob_files': 1, 'grep_files': 1, 'read_file': 7}
  B_도구명_명시 #2: 위임=False 사이클=3 {'glob_files': 1, 'grep_files': 1, 'read_file': 7}
  B_도구명_명시 #3: 위임=True 사이클=3 {'tool_search': 1, 'tool_invoke→agent_search': 1}


**읽는 법**: A는 3롤 전부 수동 스윕(read 9~15회)이다. B에서 위임이 1롤 나왔는데,
그 롤은 tool_search → tool_invoke 2단을 정확히 밟고 3사이클(도구 호출 2번)만에 끝냈다 —
위임이 되면 얼마나 싼지가 도구 호출 수로 바로 보인다. 표본이 작아 "B가 낫다"고 단정할 수는
없고, 방향이 OpenAI 권고와 일치한다는 신호로 읽는 것이 정확하다.

> **채택 공지**: 이 실측 이후 B 문구가 `cc_harness`의 기본 위임 정책으로 채택되었다
> (`prompts.DELEGATION_POLICY`). 따라서 이 노트북을 지금 재실행하면 A_기본 팔도 이미 B 문구로
> 돌게 되어 당시 조건과 달라진다 — 당시 A 문구는 위에 적힌 우회 문구다.
>
> **후속(같은 날)**: 위임 확률을 더 올리기 위해 기본 정책을 B에서 한 단계 더 확장했다 —
> "직접 훑지 마라" 금지와 조회→실행 2스텝 절차를 불릿으로 명시. 확장 문구로 같은 질문 5롤
> 실측에서 **위임 5/5** (전 롤 tool_search→tool_invoke 2단 정확, 3사이클 종료). 문구 단계별로
> 우회 0/3 → 도구명 명시 1/3 → 절차 명시 5/5. 현재 `prompts.DELEGATION_POLICY`가 이 확장 문구다.

## E2 — 병렬 emit (2-2): 넛지 문장을 더하면 병렬이 늘어나는가

In [3]:
print("=== E2 · 2-2 병렬 emit (병렬 처방 한 문장 추가 A/B) ===")
for arm, patch in [("A_기본", None),
                   ("B_미루지말라", lambda sp: sp.replace(PROMPT_PARALLEL, PROMPT_PARALLEL + PAR_EXTRA))]:
    for i in range(N):
        s, out = roll(Q_RENAME, patch=patch)
        r = summarize(out)
        r["done"] = "일치하는 내용이 없습니다" in s.fs.grep_files("authenticate")
        results.setdefault("E2_병렬", {}).setdefault(arm, []).append(r)
        print(f"  {arm} #{i+1}: 병렬={r['partition']} 사이클={r['cycles']} 완료={r['done']}")

=== E2 · 2-2 병렬 emit (병렬 처방 한 문장 추가 A/B) ===
  A_기본 #1: 병렬=True 사이클=5 완료=True
  A_기본 #2: 병렬=True 사이클=5 완료=True
  A_기본 #3: 병렬=True 사이클=5 완료=False
  B_미루지말라 #1: 병렬=True 사이클=6 완료=True
  B_미루지말라 #2: 병렬=True 사이클=6 완료=True
  B_미루지말라 #3: 병렬=True 사이클=4 완료=True


**읽는 법**: 반전 결과 — A(현재 그대로)도 3/3 병렬 emit이다. 즉 gpt-5.4-mini에게 이
작업의 병렬 emit은 원래 잘 나오는 행동이고, 메인 노트북 실행에서 파티션이 안 찍힌 것이 오히려
드문 롤이었다. 넛지 문장(B)은 이미 포화 상태라 효과를 논할 수 없다 — 채택하지 않았다.
부수 관찰: A #3은 병렬로 잘 돌고도 rename을 다 못 끝낸 채 "완료"를 선언했다(완료=False) —
모델의 완료 보고를 믿지 않고 FS를 직접 확인하는 검증 셀이 메인 노트북에 있는 이유다.

## E3 — 게이트 충돌 (2-3): @멘션 SR을 믿고 read 없이 edit하는 모델이 있는가

In [4]:
print("=== E3 · 2-3 게이트 충돌 (모델 A/B, 프롬프트 불변) ===")
for arm, model in [("A_5.4-mini", MINI), ("B_nano", "gpt-5-nano")]:
    for i in range(N):
        s, out = roll(Q_MENTION, model=model)
        r = summarize(out)
        r["debug_off"] = "DEBUG = False" in s.world.fs["/project/src/app/config.py"]["content"]
        results.setdefault("E3_게이트충돌", {}).setdefault(arm, []).append(r)
        print(f"  {arm} #{i+1}: 게이트발동={r['gate1']} 사이클={r['cycles']} DEBUG끔={r['debug_off']}")

=== E3 · 2-3 게이트 충돌 (모델 A/B, 프롬프트 불변) ===
  A_5.4-mini #1: 게이트발동=0 사이클=4 DEBUG끔=True
  A_5.4-mini #2: 게이트발동=0 사이클=3 DEBUG끔=True
  A_5.4-mini #3: 게이트발동=0 사이클=3 DEBUG끔=True
  B_nano #1: 게이트발동=0 사이클=4 DEBUG끔=True
  B_nano #2: 게이트발동=0 사이클=3 DEBUG끔=True
  B_nano #3: 게이트발동=0 사이클=5 DEBUG끔=True


**읽는 법**: 두 모델 모두 0/3 — @멘션 SR이 "다시 읽을 필요 없이 아래 내용을 참조하라"고
말해줘도, read를 건너뛰고 바로 edit하는 롤은 한 번도 없었다. 충돌(게이트1 거부 → read 복구)은
이론상 가능할 뿐 실측에서는 나타나지 않는 시나리오다. 대신 6롤 전부 주입된 메모리대로 DEBUG를
껐다 — 이 데모의 실질 증명은 "주입 정보가 행동으로 이어진다" 쪽이다.
메인 노트북 2-3 해설은 이 수치에 맞춰 작성돼 있다.

In [5]:
print("════════ 요약 ════════")
for e, arms in results.items():
    print(e)
    for arm, rolls in arms.items():
        if e == "E1_위임":
            k, lab = sum(r["delegated"] for r in rolls), "위임"
        elif e == "E2_병렬":
            k, lab = sum(r["partition"] for r in rolls), "병렬 emit"
        else:
            k, lab = sum(r["gate1"] > 0 for r in rolls), "게이트 충돌"
        avg = sum(r["cycles"] for r in rolls) / len(rolls)
        print(f"  {arm}: {lab} {k}/{len(rolls)} · 평균 사이클 {avg:.1f}")

════════ 요약 ════════
E1_위임
  A_기본: 위임 0/3 · 평균 사이클 4.0
  B_도구명_명시: 위임 1/3 · 평균 사이클 3.3
E2_병렬
  A_기본: 병렬 emit 3/3 · 평균 사이클 5.0
  B_미루지말라: 병렬 emit 3/3 · 평균 사이클 5.3
E3_게이트충돌
  A_5.4-mini: 게이트 충돌 0/3 · 평균 사이클 3.3
  B_nano: 게이트 충돌 0/3 · 평균 사이클 4.0


## 결론

1. **위임(E1)**: 기본 조건의 GPT는 위임을 택하지 않는다 (0/3). 시스템 프롬프트 정책에 도구명을
   명시하는 최소 개입만으로 위임이 나타나기 시작한다 (1/3) — "언제 쓸지는 시스템 프롬프트에,
   정확히 지시하라"는 OpenAI 규약 방향과 일치. 단, 이 하네스의 위임은 실제 CC(Agent 도구가
   tools 배열에 직접 존재, 1스텝)보다 문턱이 높은 2스텝(tool_search→tool_invoke)이라는 조건
   차이가 있다. 이 실측 후 B 문구를 `cc_harness` 기본 위임 정책으로 채택했다 — 이 정책 문장은
   CC 원문 번역이 아니라 GPT 하네스용 추가 문장이어서 원문 보존 대상이 아니기 때문이다.
   후속으로 정책을 금지·절차 명시까지 확장하자 같은 질문 5롤에서 위임 5/5 — 명시성이 오를수록
   확률이 오르는 계단(우회 0/3 → 도구명 1/3 → 절차 5/5)이 그대로 보인다.
2. **병렬 emit(E2)**: 이미 포화 (6/6) — 추가 넛지 불필요. 메인 실행에서 안 나온 건 드문 롤.
3. **게이트 충돌(E3)**: 실측 0/6 — 이론상 긴장으로만 존재. 관찰 포인트를 "충돌"이 아니라
   "주입 정보의 행동 반영"으로 잡는 것이 실측에 부합한다.

공통 교훈: **소프트(문구) 장치의 효과는 확률이고, 위치와 명시성이 그 확률을 움직인다.**
확정적 증명이 필요한 것은 전부 §1처럼 코드 게이트(하드)로 검증해야 한다.